Geocode the venues (your first Open-Meteo call)

What this is: weather is requested by latitude and longitude, but your data only has tournament names. Geocoding turns a place name into coordinates. Open-Meteo has a free geocoding endpoint that takes a name and returns matching places, each with latitude, longitude, elevation, country, region, timezone and population. It needs no API key for non-commercial use, so there's no secret to manage. Check their terms page for the current usage limits. We'll make about 120 requests, which is tiny.

What we do:

Rebuild the venue list, including the names mapped by hand.
Ask Open-Meteo for each name, and save each raw response to the bucket, write-once, with the request and fetch time recorded alongside it. This is the same pattern as the tennis files, and it's what makes the weather side reproducible, since an API can't be pinned to a commit.
Show a review table, because geocoders can pick the wrong "Charleston" or "Guadalajara". You check the picks. The code doesn't decide.

The raw tennis data only has tournament names. Need to add coordinates so weather can be requested by latitude and longitude. This notebook uses Open-Meteo's geocoding endpoint to find these coordinates

# Authentication and settings
Turn on Notebook access for secrets needed

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

# Get the venue list from BigQuery

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
TABLE = f"{PROJECT_ID}.tennis_raw.wta_matches"

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)  # safety cap: 1 GB
    return bq.query(sql, job_config=config).to_dataframe()

venues = run(f"""
SELECT LOWER(TRIM(tourney_name)) AS name_key,
       MIN(SUBSTR(tourney_date,1,4)) AS first_year,
       MAX(SUBSTR(tourney_date,1,4)) AS last_year,
       COUNT(DISTINCT tourney_id) AS editions,
       ARRAY_TO_STRING(ARRAY_AGG(DISTINCT tourney_level), ',') AS levels
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
GROUP BY name_key ORDER BY name_key
""")

# Name in the data -> the place to search for
SEARCH = {
    "australian open": "Melbourne", "melbourne 1": "Melbourne", "melbourne 2": "Melbourne",
    "gippsland trophy": "Melbourne", "yarra valley classic": "Melbourne",
    "grampians trophy": "Melbourne", "phillip island trophy": "Melbourne",
    "roland garros": "Paris", "us open": "New York", "wimbledon": "London",
    "queen's club": "London", "japan open tokyo": "Tokyo",
    "'s hertogenbosch": "'s-Hertogenbosch", "s hertogenbosch": "'s-Hertogenbosch",
    "s-hertogenbosch": "'s-Hertogenbosch", "nurnberg": "Nuremberg",
    "cluj napoca": "Cluj-Napoca", "cluj-napoca 1": "Cluj-Napoca", "cluj-napoca 2": "Cluj-Napoca",
    "adelaide 1": "Adelaide", "adelaide 2": "Adelaide", "charleston 1": "Charleston",
    "guadalajara 2": "Guadalajara", "chicago 2": "Chicago", "hua hin 2": "Hua Hin",
    "calvia": "Calvia", "st. petersburg": "Saint Petersburg", "nur-sultan": "Astana",
}
EXCLUDE = {"united cup"}   # team event with two host cities

venues["search_name"] = venues["name_key"].map(SEARCH).fillna(venues["name_key"].str.title())
venues["status"] = venues["name_key"].isin(EXCLUDE).map({True: "exclude", False: "todo"})

print(venues["status"].value_counts())
print("Places to geocode:", venues.loc[venues.status == "todo", "search_name"].nunique())

# Fetch and save raw responses
Each file is written once, so re-running the cell doesn't call the API again for anything already stored. The API's own output sits under response, and the request and fetched_at_utc fields beside it are the record of how and when it was obtained.

In [ ]:
import json, re, time
import requests
from datetime import datetime, timezone
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)

GEO_URL = "https://geocoding-api.open-meteo.com/v1/search"
GEO_PREFIX = "raw/open_meteo/geocoding"

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

def geocode(name, country=None):
    """Return the stored record for this query, fetching (and storing) it only if it is new."""
    key = slug(name) + (f"__{country}" if country else "")
    blob = bucket.blob(f"{GEO_PREFIX}/{key}.json")
    if blob.exists():
        return json.loads(blob.download_as_text())

    params = {"name": name, "count": 5, "language": "en", "format": "json"}
    if country:
        params["countryCode"] = country
    r = requests.get(GEO_URL, params=params, timeout=30)
    r.raise_for_status()

    record = {
        "request": {"url": GEO_URL, "params": params},
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "response": r.json(),          # exactly as returned by the API
    }
    blob.upload_from_string(json.dumps(record, indent=1),
                            content_type="application/json", if_generation_match=0)
    time.sleep(0.2)                    # be polite to the free API
    return record

names = sorted(venues.loc[venues.status == "todo", "search_name"].unique())
records = {n: geocode(n) for n in names}
print("Fetched or found:", len(records), "responses")

# Review Table

In [ ]:
rows = []
for name, rec in records.items():
    res = rec["response"].get("results", [])
    top = res[0] if res else {}
    second = f'{res[1].get("name")}, {res[1].get("admin1")}, {res[1].get("country_code")}' if len(res) > 1 else ""
    rows.append({"search_name": name, "found": len(res),
                 "top_name": top.get("name"), "region": top.get("admin1"),
                 "country": top.get("country_code"), "population": top.get("population"),
                 "elev_m": top.get("elevation"), "runner_up": second})

review = pd.DataFrame(rows)
pd.set_option("display.max_rows", 200, "display.width", 200)
review[["search_name", "found", "top_name", "region", "country", "population", "runner_up"]]

# Manual corrections and save venue locations

In [ ]:
# Reviewed corrections, recorded in code so the venue table can be rebuilt exactly
REFETCH = {                                   # search_name -> (text to search, country code or None)
    "Bol": ("Bol", "HR"),
    "Granby": ("Granby", "CA"),
    "Marrakech": ("Marrakesh", None),
    "Quebec City": ("Quebec", "CA"),
    "Stanford": ("Palo Alto", "US"),          # proxy: Stanford University is next to Palo Alto
    "Mallorca": ("Calvia", None),             # events are at Santa Ponsa, in Calvia (verify)
}
PICK_INDEX = {"Indian Wells": 1}              # top result is Indio; the town is the 2nd result

rows = []
for name in names:
    if name in REFETCH:
        text, country = REFETCH[name]
        rec = geocode(text, country)
        decision = f"re-searched as '{text}'" + (f" in {country}" if country else "")
    else:
        rec = records[name]
        decision = "top result"
    results = rec["response"].get("results", [])
    idx = PICK_INDEX.get(name, 0)
    if name in PICK_INDEX:
        decision = f"result #{idx + 1} chosen by hand"
    assert len(results) > idx, f"No usable result for {name}"
    r = results[idx]
    rows.append({
        "search_name": name, "place": r["name"], "region": r.get("admin1"),
        "country_code": r.get("country_code"), "latitude": r["latitude"],
        "longitude": r["longitude"], "elevation_m": r.get("elevation"),
        "timezone": r.get("timezone"), "geonames_id": r.get("id"),
        "population": r.get("population"), "decision": decision,
    })

locations = pd.DataFrame(rows)
locations[locations["search_name"].isin(list(REFETCH) + list(PICK_INDEX))]

# Checks
Are the results reasonable?

In [ ]:
print("places:", len(locations))
print("duplicate search names:", locations["search_name"].duplicated().sum())
print("missing coordinates:", locations["latitude"].isna().sum(),
      "| missing timezone:", locations["timezone"].isna().sum())
print(locations["country_code"].value_counts().head(8))
locations.sort_values("elevation_m", ascending=False).head(5)[["place", "country_code", "elevation_m"]]

# Save Table
A copy in BigQuery and a versioned file in the bucket

In [ ]:
import hashlib
from google.cloud import bigquery

# BigQuery working copy, in the clean layer
ds = bigquery.Dataset(f"{PROJECT_ID}.tennis_clean")
ds.location = REGION
bq.create_dataset(ds, exists_ok=True)

job = bq.load_table_from_dataframe(
    locations, f"{PROJECT_ID}.tennis_clean.venue_locations",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job.result()
print("BigQuery rows:", bq.get_table(f"{PROJECT_ID}.tennis_clean.venue_locations").num_rows)

# Versioned file in the bucket (write-once: changes go to v2)
csv_text = locations.to_csv(index=False)
blob = bucket.blob("reference/venues/v1/venue_locations.csv")
if blob.exists():
    print("v1 already exists, not overwriting. Save changes as v2.")
else:
    blob.upload_from_string(csv_text, content_type="text/csv", if_generation_match=0)
    print("Saved v1, sha256", hashlib.sha256(csv_text.encode()).hexdigest()[:12])

# Manifest of what was stored

In [ ]:


entries = []
for b in gcs.list_blobs(BUCKET_NAME, prefix=f"{GEO_PREFIX}/"):
    data = b.download_as_bytes()
    entries.append({"file": b.name, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()})

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/open_meteo_geocoding_{stamp}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"endpoint": GEO_URL, "files": entries}, indent=2), content_type="application/json")
print(len(entries), "files recorded in", manifest_path)

# Build the tournament editions table

What this is: one row per tournament edition (one year of one tournament) with its coordinates and a date window. It's the list of weather requests, one per row. About 870 rows are expected (877 editions in scope, minus a few for the excluded United Cup).

Date window: we only know the start date. Main-draw events run about a week, and Grand Slams and Premier Mandatory events about two. We fetch the start date plus 7 days (or plus 14 for G and PM). Fetching a few extra days costs almost nothing, and it means we never miss a match day.

# Build the table

In [ ]:
editions = run(f"""
SELECT tourney_id,
       ANY_VALUE(LOWER(TRIM(tourney_name))) AS name_key,
       COUNT(DISTINCT LOWER(TRIM(tourney_name))) AS n_names,
       ANY_VALUE(tourney_level) AS tourney_level,
       COUNT(DISTINCT tourney_level) AS n_levels,
       MIN(tourney_date) AS tourney_date,
       COUNT(DISTINCT tourney_date) AS n_dates,
       ANY_VALUE(draw_size) AS draw_size,
       COUNT(*) AS n_matches
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
GROUP BY tourney_id
""")
# Each edition must have exactly one name, level and date
assert (editions[["n_names", "n_levels", "n_dates"]] == 1).all().all()

editions = editions.merge(venues[["name_key", "search_name"]], on="name_key", how="left")
editions = editions.merge(
    locations[["search_name", "place", "country_code", "latitude", "longitude", "elevation_m", "timezone"]],
    on="search_name", how="left")

no_location = editions[editions["latitude"].isna()]
print("Editions without a location:", len(no_location), sorted(no_location["name_key"].unique()))
editions = editions.dropna(subset=["latitude"]).copy()

editions["start_date"] = pd.to_datetime(editions["tourney_date"], format="%Y%m%d")
span_days = editions["tourney_level"].map({"G": 14, "PM": 14}).fillna(7).astype(int)
editions["end_date"] = editions["start_date"] + pd.to_timedelta(span_days, unit="D")

# The archive lags by about 5 days, so never ask for dates later than that
latest_ok = pd.Timestamp.now().normalize() - pd.Timedelta(days=6)
print("Windows cut back for the archive delay:", (editions["end_date"] > latest_ok).sum())
editions["end_date"] = editions["end_date"].clip(upper=latest_ok)

editions = editions[["tourney_id", "name_key", "search_name", "place", "country_code", "tourney_level",
                     "draw_size", "start_date", "end_date", "latitude", "longitude",
                     "elevation_m", "timezone", "n_matches"]].sort_values("tourney_id").reset_index(drop=True)

# Check Table

In [ ]:
print(len(editions), "editions | tourney_id unique:", editions["tourney_id"].is_unique)
print(editions.groupby("tourney_level").size())
print(editions[["start_date", "end_date"]].agg(["min", "max"]))
print("Total days requested:", ((editions["end_date"] - editions["start_date"]).dt.days + 1).sum())
editions.sample(5, random_state=1)[["tourney_id", "place", "country_code", "start_date", "end_date", "n_matches"]]

# Save (BigQuery copy and a versioned file in bucket)

All the checks passed.

873 editions is 877 minus the 4 United Cup editions, which is exactly what we expected.
The Grand Slam count is 65, which is a neat check: 16 years × 4 = 64, minus Wimbledon 2020 (cancelled), plus the two 2026 Slams so far (Australian Open and Roland Garros). The end date of 2026-06-08 fits the latter.
7,908 days means about 190,000 hourly rows. That's small, so we'll make 873 requests, one per edition.

In [ ]:
import hashlib

to_save = editions.copy()
to_save["start_date"] = to_save["start_date"].dt.date
to_save["end_date"] = to_save["end_date"].dt.date

job = bq.load_table_from_dataframe(
    to_save, f"{PROJECT_ID}.tennis_clean.tournament_editions",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job.result()
print("BigQuery rows:", bq.get_table(f"{PROJECT_ID}.tennis_clean.tournament_editions").num_rows)

csv_text = to_save.to_csv(index=False)
blob = bucket.blob("reference/editions/v1/tournament_editions.csv")
if blob.exists():
    print("v1 already exists, not overwriting. Save changes as v2.")
else:
    blob.upload_from_string(csv_text, content_type="text/csv", if_generation_match=0)
    print("Saved v1, sha256", hashlib.sha256(csv_text.encode()).hexdigest()[:12])

A design decision to log now: pin the weather model

Open-Meteo's own docs say that data from 2017 onward uses newer, finer 9 km models, and that ERA5 (or ERA5-Land) should be selected explicitly for consistent data over decades. If we leave the model unset, the API picks a model by date, so weather from 2010 to 2016 would come from a different, coarser source than 2017 onward, and the model could learn the source change as if it were real. ERA5 has been available since 1940 at about 25 km resolution, with a delay of about 5 days. Its coarseness is fine for city-level location. So in the next step we request models=era5 for every edition.

I haven't confirmed the parameter's exact spelling in the docs' table, which the pilot fetch will check. It will compare a request with and without the setting, because results should differ for recent dates if it's working.